# Self-Query Retriever (Enterprise AI Pattern)

Self-Query Retriever is one of the **most advanced retrieval techniques** in LangChain and Enterprise RAG.

It is frequently asked in interviews because it demonstrates that you understand **metadata filtering** rather than relying only on semantic similarity.

Typical interview questions:

- What is Self-Query Retriever?
- Why use Self-Query Retriever?
- How is it different from normal retrieval?
- What is metadata filtering?
- Explain the architecture.

---

# 1. What is Self-Query Retriever?

## Definition

A **Self-Query Retriever** uses an **LLM** to understand the user's question and automatically generate:

1. **Semantic search query**
2. **Metadata filters**

It then searches the vector database using **both**.

---

## Interview Answer

> Self-Query Retriever is an Advanced RAG technique where an LLM converts a natural language question into a semantic search query and structured metadata filters. The vector database retrieves documents using both semantic similarity and metadata constraints, improving retrieval accuracy.

---

# 2. Why Do We Need Self-Query Retrieval?

Suppose we have HR documents.

| Document | Department | Year | Region |
|----------|------------|------|--------|
| Leave Policy | HR | 2025 | India |
| Leave Policy | HR | 2024 | USA |
| Payroll Policy | Finance | 2025 | India |

---

User asks

```text
Show HR Leave Policy for India in 2025
```

Traditional RAG searches only

```text
HR Leave Policy
```

It may retrieve

- India
- USA
- 2024
- 2025

Everything.

---

But Self-Query Retriever understands

```text
Department = HR

AND

Region = India

AND

Year = 2025
```

Only one document is returned.

---

# 3. Traditional RAG

```text
Question

↓

Embedding

↓

Vector Search

↓

Top K

↓

LLM
```

No metadata understanding.

---

# 4. Self-Query Retrieval

```text
Question

↓

LLM

↓

Search Query

+

Metadata Filters

↓

Vector Database

↓

Filtered Results

↓

LLM

↓

Answer
```

---

# 5. Architecture

```text
                    User Question
                           │
                           ▼
                AWS Bedrock /
               Azure OpenAI
         (Understand User Intent)
                           │
        ┌──────────────────┴──────────────────┐
        ▼                                     ▼
 Semantic Search Query                Metadata Filters
        │                                     │
        └──────────────────┬──────────────────┘
                           ▼
                OpenSearch / Qdrant /
                Azure AI Search
                           ▼
                 Filtered Documents
                           ▼
            AWS Bedrock / Azure OpenAI
                           ▼
                       Response
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Vector DB | OpenSearch / Qdrant | Azure AI Search |
| Storage | S3 | Blob Storage |
| Backend | ECS/EKS | Container Apps/AKS |

---

# 6. Metadata Example

Suppose every document has metadata.

```python
{
    "department": "HR",
    "country": "India",
    "year": 2025,
    "document_type": "Leave Policy"
}
```

Another

```python
{
    "department": "Finance",
    "country": "USA",
    "year": 2024,
    "document_type": "Payroll"
}
```

---

# 7. User Question

```text
Show HR leave policy for India published in 2025
```

LLM converts it into

Search Query

```text
Leave Policy
```

Metadata

```text
department = HR

country = India

year = 2025
```

---

Retriever executes

```text
Vector Search

+

Metadata Filter
```

---

# 8. Enterprise Flow

```text
User

↓

FastAPI

↓

LangGraph

↓

Self Query Retriever

↓

Generate Metadata Filters

↓

OpenSearch

↓

Filtered Documents

↓

Bedrock

↓

Answer
```

---

# 9. LangChain Example

```python
# ==========================================================
# STEP 1 : Create Bedrock LLM
#
# Purpose:
# The LLM converts natural language into:
# - Semantic search query
# - Metadata filters
# ==========================================================

from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name="us-east-1"
)


# ==========================================================
# STEP 2 : Describe Metadata Fields
#
# Purpose:
# Tell the LLM which metadata fields exist.
# ==========================================================

from langchain.chains.query_constructor.schema import AttributeInfo

metadata_field_info = [

    AttributeInfo(
        name="department",
        description="HR, Finance, IT",
        type="string"
    ),

    AttributeInfo(
        name="country",
        description="India, USA, UK",
        type="string"
    ),

    AttributeInfo(
        name="year",
        description="Document publication year",
        type="integer"
    ),
]


# ==========================================================
# STEP 3 : Create Self Query Retriever
#
# Purpose:
# The retriever generates semantic queries
# and metadata filters automatically.
# ==========================================================

from langchain.retrievers.self_query.base import SelfQueryRetriever

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents="HR policy documents",
    metadata_field_info=metadata_field_info,
)


# ==========================================================
# STEP 4 : Search
#
# Purpose:
# The retriever automatically applies
# metadata filters.
# ==========================================================

docs = retriever.invoke(
    "Show HR leave policy for India in 2025"
)

for doc in docs:
    print(doc.page_content)
```

> **Production Note:** This requires a vector store that supports metadata filtering (e.g., **OpenSearch**, **Qdrant**, **Pinecone**, or **Azure AI Search**) and documents indexed with structured metadata.

---

# 10. What Happens Internally?

User asks

```text
HR Leave Policy India 2025
```

LLM produces

```text
Semantic Query

↓

Leave Policy
```

Metadata

```text
Department = HR

Country = India

Year = 2025
```

Database executes

```text
Vector Search

AND

Metadata Filter
```

---

# 11. Advantages

✅ More accurate retrieval

✅ Less irrelevant context

✅ Faster search

✅ Better enterprise filtering

✅ Lower LLM cost

---

# 12. Disadvantages

❌ Requires metadata

❌ More complex indexing

❌ LLM required before retrieval

❌ Slightly higher latency

---

# 13. Best Practices

✅ Store rich metadata during document ingestion

Examples

- Department
- Country
- Product
- Version
- Date
- Document Type
- Author

---

✅ Use metadata filters before reranking.

---

✅ Combine with Hybrid Search.

---

# 14. Common Mistakes

❌ No metadata

❌ Inconsistent metadata values

❌ Missing dates

❌ No document categories

---

# 15. Self-Query vs Traditional RAG

| Traditional RAG | Self-Query Retriever |
|-----------------|----------------------|
| Semantic Search | Semantic + Metadata |
| No Filters | Automatic Filters |
| More irrelevant results | More precise results |
| Simpler | Smarter |

---

# 16. Self-Query vs Multi-Query

| Multi-Query | Self-Query |
|-------------|------------|
| Generates multiple queries | Generates one query + metadata filters |
| Improves recall | Improves precision |
| Multiple searches | Single filtered search |
| Higher search cost | Lower search cost |

---

# 17. Real Enterprise Example

## HR Assistant

Question

```text
Show Leave Policy for India in 2025
```

Generated filters

```text
Department = HR

Country = India

Year = 2025
```

Only the correct policy document is retrieved.

---

## Healthcare Assistant

Question

```text
Show diabetes treatment guidelines for adults published after 2023
```

Metadata filters

```text
Disease = Diabetes

PatientType = Adult

Year > 2023
```

The retriever searches only documents matching those criteria before applying semantic similarity.

---

# 18. Production Architecture

```text
User

↓

FastAPI

↓

JWT

↓

LangGraph

↓

Self Query Retriever

↓

LLM Creates Metadata Filters

↓

Hybrid Search

↓

OpenSearch / Azure AI Search

↓

Reranker

↓

Context Compression

↓

Bedrock / Azure OpenAI

↓

Redis

↓

Response
```

---

# 19. Common Interview Questions

### Q1. Why Self-Query Retriever?

To improve retrieval precision by combining semantic search with automatic metadata filtering.

---

### Q2. What is metadata?

Structured information about a document.

Examples

- Department
- Country
- Product
- Version
- Author
- Date

---

### Q3. Can Self-Query Retriever work without metadata?

Not effectively.

Without metadata, it behaves similarly to a standard semantic retriever.

---

### Q4. Can Self-Query Retriever be combined with Hybrid Search?

Yes.

A common production pipeline is:

```text
LLM

↓

Metadata Filters

↓

Hybrid Search

↓

Reranker

↓

LLM
```

---

### Q5. Where is it used?

- HR Assistants
- Banking
- Insurance
- Healthcare
- Legal AI
- Compliance systems
- Product documentation

---

# 20. Traditional vs Hybrid vs Self-Query

| Feature | Traditional RAG | Hybrid Search | Self-Query Retriever |
|---------|------------------|---------------|----------------------|
| Semantic Search | ✅ | ✅ | ✅ |
| Keyword Search | ❌ | ✅ | Optional |
| Metadata Filtering | ❌ | Manual | ✅ Automatic |
| LLM Before Retrieval | ❌ | ❌ | ✅ |
| Retrieval Precision | Medium | High | Very High |

---

# 21. EPAM Senior Answer (3 Minutes)

> "Self-Query Retriever is an Advanced RAG technique that uses an LLM to convert a natural language question into both a semantic search query and structured metadata filters. During document ingestion, enterprise documents are indexed with metadata such as department, country, document type, version, and publication date. When a user submits a query like 'Show the HR leave policy for India published in 2025', the LLM extracts both the search intent and metadata constraints. The vector database—such as Amazon OpenSearch, Qdrant, or Azure AI Search—performs semantic similarity search while applying the metadata filters, significantly improving retrieval precision. The retrieved documents are then optionally reranked and passed to Amazon Bedrock or Azure OpenAI for response generation. In production, I typically combine Self-Query Retrieval with Hybrid Search, reranking, and context compression to build highly accurate enterprise RAG systems."